# DE-06 — Data Quality & Contracts

**Dataset:** `data/loan_data_06.csv`

This notebook distinguishes technical and business rules, validates schema/type/completeness/uniqueness/validity/reconciliation, routes invalid data, publishes a quality score, and demonstrates STOP, WARN, and QUARANTINE behavior with ownership and rule versioning.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

DATA_FILE = ROOT / "data" / "loan_data_06.csv"
assert DATA_FILE.exists(), f"Dataset not found: {DATA_FILE}"

raw = pd.read_csv(DATA_FILE)
print(f"Dataset: {DATA_FILE.name}")
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
print(raw.head(3).to_string(index=False))

## Learning Content

- **Technical rules** protect processability: schema, type, key presence, and parseability.
- **Business rules** protect meaning: permitted status, positive amount, and policy constraints.
- **Completeness** measures required values.
- **Uniqueness** protects declared keys.
- **Validity** checks domains and ranges.
- **Reconciliation** compares counts and measures across boundaries.
- **Quarantine** isolates invalid records for investigation.
- **Tolerance** allows explicitly accepted non-critical variance.
- **Severity** determines whether a failure stops, warns, or quarantines.
- A data contract records schema, owner, consumers, rule version, and change policy.

In [ ]:
contract = {
    "name": "loan_application_contract",
    "version": "1.0.0",
    "owner": "Loan Data Product Owner",
    "producer": "Loan Operations",
    "consumers": ["Risk Analytics", "BI"],
    "business_key": "Loan_ID",
    "required_columns": [
        "Loan_ID", "ApplicantIncome", "CoapplicantIncome",
        "LoanAmount", "Loan_Status",
    ],
    "allowed_status": ["Y", "N"],
    "change_policy": "Backward-compatible additions require notice; breaking changes require approval",
}

missing_columns = set(contract["required_columns"]).difference(raw.columns)
assert not missing_columns
print("Contract:", contract)

## Hands-on / Demonstration

The source partition may not contain a critical invalid record. To prove quarantine routing, the next cell appends one clearly labeled synthetic record. The original CSV and `raw` DataFrame remain unchanged.

In [ ]:
controlled_bad = raw.iloc[[0]].copy()
controlled_bad["Loan_ID"] = "TRAINING_INVALID_001"
controlled_bad["ApplicantIncome"] = -100
controlled_bad["LoanAmount"] = 0
controlled_bad["Loan_Status"] = "INVALID"

candidate = pd.concat([raw, controlled_bad], ignore_index=True)
print("Source rows:", len(raw))
print("Candidate rows including controlled invalid record:", len(candidate))

In [ ]:
def evaluate_quality(frame: pd.DataFrame):
    evaluated = frame.copy()
    for column in ["ApplicantIncome", "CoapplicantIncome", "LoanAmount",
                   "Loan_Amount_Term", "Credit_History"]:
        evaluated[column] = pd.to_numeric(evaluated[column], errors="coerce")

    evaluated["technical_error"] = ""
    evaluated.loc[evaluated["Loan_ID"].isna(), "technical_error"] += "missing Loan_ID; "
    evaluated.loc[evaluated["Loan_ID"].duplicated(keep=False), "technical_error"] += "duplicate Loan_ID; "
    evaluated.loc[evaluated["LoanAmount"].isna(), "technical_error"] += "unparseable LoanAmount; "

    evaluated["business_error"] = ""
    evaluated.loc[evaluated["ApplicantIncome"] < 0, "business_error"] += "negative applicant income; "
    evaluated.loc[evaluated["CoapplicantIncome"] < 0, "business_error"] += "negative coapplicant income; "
    evaluated.loc[evaluated["LoanAmount"] <= 0, "business_error"] += "non-positive loan amount; "
    evaluated.loc[~evaluated["Loan_Status"].isin(["Y", "N"]), "business_error"] += "invalid loan status; "

    evaluated["error_reason"] = evaluated["technical_error"] + evaluated["business_error"]
    evaluated["is_valid"] = evaluated["error_reason"].eq("")
    return evaluated

evaluated = evaluate_quality(candidate)
accepted = evaluated[evaluated["is_valid"]].copy()
quarantine = evaluated[~evaluated["is_valid"]].copy()

print("Accepted rows:", len(accepted))
print("Quarantined rows:", len(quarantine))
print(quarantine[["Loan_ID", "error_reason"]].to_string(index=False))

### Quality assertions and published score

Warnings for optional attributes do not block publication. Critical failures are quarantined before the accepted dataset is served.

In [ ]:
quality_results = pd.DataFrame([
    ["schema_required_columns", not missing_columns, "CRITICAL", 0.0],
    ["loan_id_complete", candidate["Loan_ID"].notna().all(), "CRITICAL", 0.0],
    ["loan_id_unique", candidate["Loan_ID"].is_unique, "CRITICAL", 0.0],
    ["loan_status_valid", candidate["Loan_Status"].isin(["Y", "N"]).all(), "CRITICAL", 0.0],
    ["loan_amount_positive", pd.to_numeric(candidate["LoanAmount"], errors="coerce").gt(0).all(), "CRITICAL", 0.0],
    ["gender_complete", candidate["Gender"].notna().mean() >= 0.95, "WARNING", 0.05],
], columns=["rule_name", "passed", "severity", "tolerance"])

quality_score = quality_results["passed"].mean() * 100
critical_failures = quality_results[
    (~quality_results["passed"]) & quality_results["severity"].eq("CRITICAL")
]

print(quality_results.to_string(index=False))
print(f"Published quality score: {quality_score:.2f}%")
print("Critical failures:", len(critical_failures))

### Explicit STOP, WARN, and QUARANTINE behavior

In [ ]:
def apply_failure_mode(frame: pd.DataFrame, mode: str):
    mode = mode.upper()
    checked = evaluate_quality(frame)
    invalid = checked[~checked["is_valid"]]
    valid = checked[checked["is_valid"]]

    if mode == "STOP" and not invalid.empty:
        return {"status": "STOPPED", "published": 0, "quarantined": len(invalid)}
    if mode == "WARN":
        return {"status": "PUBLISHED_WITH_WARNING", "published": len(checked), "quarantined": 0}
    if mode == "QUARANTINE":
        return {"status": "PUBLISHED_VALID_ONLY", "published": len(valid), "quarantined": len(invalid)}
    raise ValueError("Mode must be STOP, WARN, or QUARANTINE")

mode_results = {
    mode: apply_failure_mode(candidate, mode)
    for mode in ["STOP", "WARN", "QUARANTINE"]
}
print(mode_results)

assert mode_results["STOP"]["published"] == 0
assert mode_results["WARN"]["published"] == len(candidate)
assert mode_results["QUARANTINE"]["published"] == len(accepted)
assert mode_results["QUARANTINE"]["quarantined"] == len(quarantine)

## Enterprise Control

Quality failures need explicit stop, warn, or quarantine behavior agreed with data owners.

Each rule must have:

- A named owner and consumer impact.
- A version and effective date.
- Severity and tolerance.
- A deterministic failure action.
- Evidence: run ID, counts, failing keys, and timestamps.

In [ ]:
quality_evidence = {
    "contract": contract["name"],
    "rule_version": contract["version"],
    "owner": contract["owner"],
    "source_rows": len(raw),
    "evaluated_rows": len(candidate),
    "accepted_rows": len(accepted),
    "quarantined_rows": len(quarantine),
    "quality_score": round(quality_score, 2),
    "reconciled": len(candidate) == len(accepted) + len(quarantine),
}

assert quality_evidence["reconciled"]
assert "TRAINING_INVALID_001" in set(quarantine["Loan_ID"])
print("DE-06 evidence:", quality_evidence)